In [32]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pandas as pd
import muon as mu
import scanpy as sc
import scirpy as ir
np.random.seed(42)
import random
random.seed(42)
import sys
# sys.path.append(r"E:\Python code\Machine learning\JupyterNote\Bio_CRC\Data processing\functions")
sys.path.append(r"/ihome/ylee/yiz133/Code/Data processing/functions/")
import mdata_utils 
import TCR_embedings

In [33]:
from sklearn.preprocessing import StandardScaler
from scipy.sparse import issparse

In [34]:
# path = r"/ix1/ylee/Yifan_Zhang/Code_data/Tumor/GSE139555_2019/data/Processed/"
# filename = "T_DE_per_sample_TCRemb.h5mu"
# DATA_PATH = path + filename

# mdata_ori = mu.read(DATA_PATH)

In [54]:
mdata_raw = mu.read("/ix1/ylee/Yifan_Zhang/Code_data/Tumor/GSE139555_2019/data/Raw/wu2020.h5mu")
mdata_raw

MuData object with n_obs × n_vars = 161015 × 30727
  2 modalities
    gex:	141623 x 30727
      obs:	'cluster_orig', 'patient', 'sample', 'source'
      uns:	'cluster_orig_colors'
      obsm:	'X_umap_orig'
    airr:	123134 x 0
      obs:	'high_confidence', 'is_cell', 'clonotype_orig'
      obsm:	'airr'

In [62]:
airr = sc.read("/ix1/ylee/Yifan_Zhang/Code_data/Tumor/GSE139555_2019/data/Raw/vdjdb.h5ad")
airr

AnnData object with n_obs × n_vars = 139744 × 0
    obs: 'antigen.epitope', 'antigen.gene', 'antigen.species', 'meta.cell.subset', 'meta.clone.id', 'meta.donor.MHC', 'meta.donor.MHC.method', 'meta.epitope.id', 'meta.replica.id', 'meta.structure.id', 'meta.study.id', 'meta.subject.cohort', 'meta.subject.id', 'meta.tissue', 'method.frequency', 'method.identification', 'method.sequencing', 'method.singlecell', 'method.verification', 'mhc.a', 'mhc.b', 'mhc.class', 'reference.id', 'species'
    uns: 'DB', 'chain_indices', 'scirpy_version'
    obsm: 'airr', 'chain_indices'

In [71]:
num_different_cats = airr.obs['antigen.species'].nunique()
print(f"Number of different cats: {num_different_cats}")

num_different_cats = airr.obs['antigen.epitope'].nunique()
print(f"Number of different cats : {num_different_cats}")

num_different_cats = airr.obs['meta.epitope.id'].nunique()
print(f"Number of different cats : {num_different_cats}")

num_different_cats = airr.obs['meta.clone.id'].nunique()
print(f"Number of different cats : {num_different_cats}")

Number of different cats: 49
Number of different cats : 1785
Number of different cats : 222
Number of different cats : 37879


In [72]:
airr.obs['antigen.epitope']

cell_id
0                    GILGFVFTL
1                    GILGFVFTL
2                    GILGFVFTL
3                    GILGFVFTL
4                    GILGFVFTL
                  ...         
139739    FRDYVDRFYKTLRAEQASQE
139740    FRDYVDRFYKTLRAEQASQE
139741    FRDYVDRFYKTLRAEQASQE
139742    FRDYVDRFYKTLRAEQASQE
139743    FRDYVDRFYKTLRAEQASQE
Name: antigen.epitope, Length: 139744, dtype: category
Categories (1785, object): ['AAFKRSCLK', 'AAGIGILTV', 'AAKMYAFTL', 'AALALLLLDRLNQLE', ..., 'YYKKDNSYF', 'YYQLYSTQL', 'YYRYNLPTM', 'YYTSNPTTF']

In [68]:
airr.obs['meta.epitope.id']

cell_id
0         M158–66
1         M158–66
2         M158–66
3         M158–66
4         M158–66
           ...   
139739        NaN
139740        NaN
139741        NaN
139742        NaN
139743        NaN
Name: meta.epitope.id, Length: 139744, dtype: category
Categories (222, object): ['7-16V', '10-19R', '5316', '13701', ..., 'p53R175H', 'pp65', 'pp65(495)', 'vivo']

In [55]:
query_airr = mdata_raw['airr']
query_airr

AnnData object with n_obs × n_vars = 123134 × 0
    obs: 'high_confidence', 'is_cell', 'clonotype_orig'
    obsm: 'airr'

In [73]:
query_airr.obs['clonotype_orig']

RT2_AAACCTGAGGTAGCCA-1     renal2.tnb.C1
RT2_AAACCTGCAGGTCCAC-1     renal2.tnb.C2
RT2_AAACCTGCATCCGTGG-1     renal2.tnb.C1
RT2_AAACCTGCATCGTCGG-1     renal2.tnb.C3
RT2_AAACCTGTCTGAGTGT-1     renal2.tnb.C4
                               ...      
CT2_TTTGGTTGTAGCTAAA-1               NaN
CT2_TTTGGTTGTGCCTTGG-1    colon2.tn.C954
CT2_TTTGGTTTCACTTACT-1    colon2.tn.C955
CT2_TTTGGTTTCAGCTTAG-1               NaN
CT2_TTTGTCAGTATTCGTG-1    colon2.tn.C765
Name: clonotype_orig, Length: 123134, dtype: category
Categories (55260, object): ['colon1.tn.C1', 'colon1.tn.C2', 'colon1.tn.C3', 'colon1.tn.C4', ..., 'renal3.tnb.C2179', 'renal3.tnb.C2180', 'renal3.tnb.C2181', 'renal3.tnb.C2182']

In [46]:
query_airr.obsm['airr'].fields

['c_call',
 'cdr3',
 'cdr3_aa',
 'consensus_count',
 'd_call',
 'd_cigar',
 'duplicate_count',
 'germline_alignment',
 'j_call',
 'j_cigar',
 'junction',
 'junction_aa',
 'locus',
 'productive',
 'rev_comp',
 'sequence',
 'sequence_alignment',
 'sequence_id',
 'v_call',
 'v_cigar']

In [56]:
ir.pp.index_chains(query_airr)
ir.tl.chain_qc(query_airr)
meta_airr = ir.get.airr(query_airr, ["junction_aa", "v_call", "j_call"] ,  ('VJ_1', 'VDJ_1'))
query_airr.obs = query_airr.obs.join(meta_airr)

In [ ]:
#### VDJdb

In [63]:
ir.pp.index_chains(airr)
ir.tl.chain_qc(airr)
meta_airr2 = ir.get.airr(airr, ["junction_aa", "v_call", "j_call"] ,  ('VJ_1', 'VDJ_1'))
airr.obs = airr.obs.join(meta_airr2)

In [ ]:
# KEY_COLS = [
#     'VJ_1_junction_aa',
#     'VDJ_1_junction_aa',
    
#     # "VJ_1_cdr3_aa",
#     # "VDJ_1_cdr3_aa",
#      # "VJ_1_v_call", "VJ_1_j_call",
#      # "VDJ_1_v_call", "VDJ_1_j_call",
# ]

# def make_key(obs, cols, sep="|", na="<NA>"):
#     missing = [c for c in cols if c not in obs.columns]
#     if missing:
#         raise KeyError(f"Missing columns: {missing}")
#     return obs[cols].astype("string").fillna(na).agg(sep.join, axis=1)

# # reference clonotype keys
# ref_keys = set(make_key(airr.obs, KEY_COLS))

# # query keys on the airr modality, then membership test
# q_keys = make_key(query_airr.obs, KEY_COLS)
# in_ref = q_keys.isin(ref_keys)          # bool Series, indexed by airr obs_names

# # write to the modality and to the global mdata.obs (aligned by index)
# query_airr.obs["in_ref"] = in_ref.values
# query_airr.obs["in_ref"] = in_ref.reindex(query_airr.obs_names).fillna(False).astype(bool)

# query_airr.obs["in_ref"].value_counts()

In [64]:
## OR logic
# --- 1. VDJdb reference: HomoSapiens TCRs, CMV/EBV/InfluenzaA epitopes ---
ref = airr.obs
ref = ref[
    (ref["species"] == "HomoSapiens")
    & (ref["antigen.species"].isin(["CMV", "EBV", "InfluenzaA"]))
]

# pool all CDR3-aa (both chains) into a single set, as the paper does ("any CDR3")
ref_cdr3 = set(
    pd.concat([ref["VJ_1_junction_aa"], ref["VDJ_1_junction_aa"]])
      .dropna().astype("string")
)
print(f"VDJdb reference: {len(ref)} entries, {len(ref_cdr3)} unique CDR3-aa")

# --- 2. Query: collapse to clonotype level ---
q = query_airr.obs
q_long = pd.concat([
    q[["clonotype_orig", "VJ_1_junction_aa"]].rename(columns={"VJ_1_junction_aa": "cdr3"}),
    q[["clonotype_orig", "VDJ_1_junction_aa"]].rename(columns={"VDJ_1_junction_aa": "cdr3"}),
]).dropna(subset=["cdr3"])
q_long["cdr3"] = q_long["cdr3"].astype("string")
q_long["hit"] = q_long["cdr3"].isin(ref_cdr3)

# a clonotype matches if ANY of its CDR3 sequences hit
clono_match = q_long.groupby("clonotype_orig")["hit"].any()

# --- 3. Report at the clonotype level (comparable to the paper's ~3%) ---
n_clono, n_match = clono_match.size, int(clono_match.sum())
print(f"{n_match} / {n_clono} clonotypes match VDJdb ({n_match / n_clono:.1%})")

# --- 4. Propagate the label back to cells ---
query_airr.obs["vdjdb_match"] = (
    query_airr.obs["clonotype_orig"].map(clono_match).fillna(False).astype(bool)
)
query_airr.obs["vdjdb_match"].value_counts()

VDJdb reference: 67852 entries, 68767 unique CDR3-aa
6443 / 55260 clonotypes match VDJdb (11.7%)


vdjdb_match
False    110701
True      12433
Name: count, dtype: int64